# Kinase Panel Docking & Selectivity Analysis

**10 structurally diverse kinases** from 5 kinase groups — batch docking, cross-docking selectivity, ProLIF interaction analysis, and comparative visualization.

| # | Kinase | Group | PDB | Family | Relevance |
|---|--------|-------|-----|--------|-----------|
| 1 | EGFR | TK (RTK) | 1M17 | ErbB | NSCLC — Erlotinib |
| 2 | ABL1 | TK (NRTK) | 1IEP | Abl | CML — Imatinib |
| 3 | JAK2 | TK (NRTK) | 3FUP | JAK | MPN — Ruxolitinib |
| 4 | CDK2 | CMGC | 1H1S | CDK | Cell cycle — Roscovitine |
| 5 | p38α MAPK | CMGC | 3GCP | MAPK | Inflammation — Doramapimod |
| 6 | GSK3β | CMGC | 1Q5K | GSK | Alzheimer's — Staurosporine |
| 7 | BRAF | TKL | 3OG7 | RAF | Melanoma — Vemurafenib |
| 8 | ALK | TKL | 2XP2 | ALK | NSCLC — Crizotinib |
| 9 | AKT1 | AGC | 3CQW | AKT | Cancer — Inhibitor VIII |
| 10 | AURKA | Other | 3E5A | Aurora | Mitosis — Hesperadin |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/ddtut1/blob/main/Kinase_Panel_Docking_Colab.ipynb)

---
## 1. Environment Setup

In [ ]:
#@title 1-1. Install Python Packages {display-mode: "form"}
!pip install -q rdkit gemmi
!pip install -q meeko gemmi vina
!pip install -q openbabel-wheel
!pip install -q pdbfixer openmm
!pip install -q pymol-open-source
!pip install -q py3Dmol
!pip install -q MDAnalysis
!pip install -q prolif
!pip install -q seaborn scipy scikit-learn
print('\n=== All packages installed ===')

### 1-2. Install Binary Tools (smina, ADFRsuite)

도킹 엔진(smina)과 구조 변환 도구(ADFRsuite)의 바이너리를 다운로드합니다.


In [ ]:
#@title 1-2. Install Binary Tools (smina, ADFRsuite) {display-mode: "form"}
import os, stat, urllib.request, tarfile, shutil

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)

def _chmod_x(path):
    st = os.stat(path)
    os.chmod(path, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# smina
smina_path = f'{BIN_DIR}/smina'
if not os.path.exists(smina_path):
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    _chmod_x(smina_path)

# ADFRsuite
ADFR_DIR = f'{BIN_DIR}/ADFRsuite'
if not os.path.exists(ADFR_DIR):
    print('Downloading ADFRsuite...')
    adfr_tar = '/tmp/ADFRsuite.tar.gz'
    urllib.request.urlretrieve('https://ccsb.scripps.edu/adfr/download/1038/', adfr_tar)
    with tarfile.open(adfr_tar) as tar:
        tar.extractall('/tmp/')
    adfr_ext = [d for d in os.listdir('/tmp/') if d.startswith('ADFRsuite')][0]
    try:
        os.chdir(f'/tmp/{adfr_ext}')
        !echo 'Y' | ./install.sh -d {ADFR_DIR} -c 0 > /dev/null 2>&1
    except Exception as e:
        print(f'ADFRsuite install warning: {e}')
    finally:
        os.chdir('/content')

for tool in ['prepare_receptor', 'prepare_ligand']:
    wrapper = f'{BIN_DIR}/{tool}'
    adfr_tool = f'{ADFR_DIR}/bin/{tool}'
    if os.path.exists(adfr_tool) and not os.path.exists(wrapper):
        with open(wrapper, 'w') as f:
            f.write(f'#!/bin/bash\n{adfr_tool} "$@"\n')
        _chmod_x(wrapper)

os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('=== Binary tools ready ===')

### 1-3. Import Libraries

분자 도킹 파이프라인에 필요한 라이브러리를 불러옵니다.


In [ ]:
#@title 1-3. Import Libraries {display-mode: "form"}
from pymol import cmd
import py3Dmol
from vina import Vina
from openbabel import pybel

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools, DataStructs

from pdbfixer import PDBFixer
from openmm.app import PDBFile

import MDAnalysis as mda
from MDAnalysis.coordinates import PDB
import prolif as plf
from prolif.plotting.network import LigNetwork

import os, sys, glob, random, warnings, traceback
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from collections import defaultdict

warnings.filterwarnings('ignore')
BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
print('All libraries imported.')

---
## 2. Kinase Panel Definition

In [ ]:
#@title 2-1. Define 10 Kinases {display-mode: "form"}

KINASES = [
    # Tyrosine Kinases (TK)
    {'name': 'EGFR',  'pdb': '1M17', 'chain': 'A', 'group': 'TK',
     'family': 'ErbB (RTK)',  'desc': 'Epidermal growth factor receptor',
     'disease': 'NSCLC', 'drug': 'Erlotinib'},

    {'name': 'ABL1',  'pdb': '1IEP', 'chain': 'A', 'group': 'TK',
     'family': 'Abl (NRTK)',  'desc': 'Abelson tyrosine kinase',
     'disease': 'CML', 'drug': 'Imatinib'},

    {'name': 'JAK2',  'pdb': '3FUP', 'chain': 'A', 'group': 'TK',
     'family': 'JAK (NRTK)',  'desc': 'Janus kinase 2',
     'disease': 'MPN', 'drug': 'Ruxolitinib'},

    # CMGC group
    {'name': 'CDK2',  'pdb': '1H1S', 'chain': 'A', 'group': 'CMGC',
     'family': 'CDK',  'desc': 'Cyclin-dependent kinase 2',
     'disease': 'Cancer', 'drug': 'Roscovitine'},

    {'name': 'p38a',  'pdb': '3GCP', 'chain': 'A', 'group': 'CMGC',
     'family': 'MAPK', 'desc': 'p38 MAP kinase alpha',
     'disease': 'Inflammation', 'drug': 'Doramapimod (BIRB-796)'},

    {'name': 'GSK3b', 'pdb': '1Q5K', 'chain': 'A', 'group': 'CMGC',
     'family': 'GSK',  'desc': 'Glycogen synthase kinase 3 beta',
     'disease': 'Alzheimer\'s', 'drug': 'Staurosporine'},

    # Tyrosine Kinase-Like (TKL)
    {'name': 'BRAF',  'pdb': '3OG7', 'chain': 'A', 'group': 'TKL',
     'family': 'RAF',  'desc': 'B-Raf proto-oncogene kinase V600E',
     'disease': 'Melanoma', 'drug': 'Vemurafenib (PLX4032)'},

    {'name': 'ALK',   'pdb': '2XP2', 'chain': 'A', 'group': 'TKL',
     'family': 'ALK',  'desc': 'Anaplastic lymphoma kinase',
     'disease': 'NSCLC', 'drug': 'Crizotinib'},

    # AGC group
    {'name': 'AKT1',  'pdb': '3CQW', 'chain': 'A', 'group': 'AGC',
     'family': 'AKT',  'desc': 'RAC-alpha serine/threonine kinase',
     'disease': 'Cancer', 'drug': 'Inhibitor VIII'},

    # Other
    {'name': 'AURKA', 'pdb': '3E5A', 'chain': 'A', 'group': 'Other',
     'family': 'Aurora', 'desc': 'Aurora kinase A',
     'disease': 'Cancer', 'drug': 'Hesperadin'},
]

# Display
panel_df = pd.DataFrame(KINASES)
panel_df.index += 1
panel_df[['name', 'pdb', 'group', 'family', 'disease', 'drug']]

---
## 3. Utility & Pipeline Functions

Refined, reusable functions for the full docking pipeline.

In [ ]:
#@title 3-1. Utility Functions {display-mode: "form"}

def getbox(selection='sele', extending=6.0):
    """
    Calculate Vina docking box from a PyMOL selection.

    Parameters
    ----------
    selection : str
        PyMOL selection name (e.g., 'lig').
    extending : float
        Padding (Angstrom) around the selection bounding box.

    Returns
    -------
    center : dict  {'center_x', 'center_y', 'center_z'}
    size   : dict  {'size_x',   'size_y',   'size_z'}
    """
    ([minX, minY, minZ], [maxX, maxY, maxZ]) = cmd.get_extent(selection)
    pad = float(extending)
    minX -= pad; minY -= pad; minZ -= pad
    maxX += pad; maxY += pad; maxZ += pad
    center = {
        'center_x': (maxX + minX) / 2,
        'center_y': (maxY + minY) / 2,
        'center_z': (maxZ + minZ) / 2,
    }
    size = {
        'size_x': maxX - minX,
        'size_y': maxY - minY,
        'size_z': maxZ - minZ,
    }
    return center, size


def fix_protein(filename, output, addHs_pH=7.4, try_renumber=False):
    """
    Fix missing residues/atoms and add hydrogens with PDBFixer.

    Parameters
    ----------
    filename : str   Input PDB (cleaned, no ligand).
    output   : str   Output PDB path.
    addHs_pH : float pH for protonation.
    try_renumber : bool  Attempt to restore original residue numbering.
    """
    fixer = PDBFixer(filename=filename)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(True)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(addHs_pH)
    with open(output, 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    if try_renumber:
        try:
            orig = mda.Universe(filename)
            fixed = mda.Universe(output)
            for idx, res in enumerate(fixed.residues):
                res.resid = orig.residues[idx].resid
            w = PDB.PDBWriter(filename=output)
            w.write(fixed)
            w.close()
        except Exception:
            pass  # keep PDBFixer numbering


def pdbqt_to_sdf(pdbqt_file, output):
    """
    Convert Vina PDBQT output to SDF with pose/score metadata.
    """
    mols = list(pybel.readfile(filename=pdbqt_file, format='pdbqt'))
    out = pybel.Outputfile(filename=output, format='sdf', overwrite=True)
    for mol in mols:
        mol.data['Pose'] = mol.data.get('MODEL', '?')
        mol.data['Score'] = mol.data.get('REMARK', '').split()[2] if 'REMARK' in mol.data else '0'
        for key in ['MODEL', 'REMARK', 'TORSDO']:
            mol.data.pop(key, None)
        out.write(mol)
    out.close()


print('Utility functions ready: getbox, fix_protein, pdbqt_to_sdf')

### 3-2. Pipeline Functions

도킹 파이프라인에서 사용하는 유틸리티 함수를 정의합니다.


In [ ]:
#@title 3-2. Pipeline Functions {display-mode: "form"}

class DockingPipeline:
    """
    End-to-end molecular docking pipeline for a single kinase target.

    Usage
    -----
    >>> pipe = DockingPipeline('EGFR', '1M17', chain='A', work_dir='/content/docking/EGFR')
    >>> pipe.prepare()          # fetch PDB, clean, add H, make PDBQT
    >>> pipe.dock_native()      # dock co-crystallized ligand (Vina or smina)
    >>> pipe.dock_smiles(smi, name)  # dock arbitrary SMILES
    >>> pipe.dock_sdf(sdf_path)     # dock ligands from SDF file
    >>> df = pipe.get_scores()  # scores DataFrame
    """

    def __init__(self, name, pdb_code, chain='A', work_dir=None, extending=7.0):
        self.name = name
        self.pdb = pdb_code.lower()
        self.chain = chain
        self.extending = extending
        self.work_dir = work_dir or f'/content/docking/{name}'
        os.makedirs(self.work_dir, exist_ok=True)

        # File paths (set during prepare)
        self.clean_pdb = None
        self.lig_mol2 = None
        self.prot_H = None
        self.lig_H = None
        self.rec_qt = None
        self.lig_qt = None
        self.center = None
        self.size = None
        self.docking_results = {}  # label -> sdf_path
        self.prepared = False

    # ---------- helpers ----------
    def _path(self, filename):
        return os.path.join(self.work_dir, filename)

    # ---------- 1. PDB fetch & clean ----------
    def _fetch_clean(self):
        """Fetch PDB, isolate chain, extract protein + ligand."""
        cmd.delete('all')
        self.clean_pdb = self._path(f'{self.pdb}_clean.pdb')
        self.lig_mol2  = self._path(f'{self.pdb}_lig.mol2')

        cmd.fetch(code=self.pdb, type='pdb', path=self.work_dir)
        cmd.remove(f'not chain {self.chain}')
        cmd.select('Prot', 'polymer.protein')
        cmd.select('Lig',  'organic')

        n_lig_atoms = cmd.count_atoms('Lig')
        if n_lig_atoms < 5:
            # Fallback: try all chains for ligand
            cmd.delete('all')
            cmd.fetch(code=self.pdb, type='pdb', path=self.work_dir)
            cmd.select('Prot', f'chain {self.chain} and polymer.protein')
            cmd.select('Lig',  'organic')
            n_lig_atoms = cmd.count_atoms('Lig')

        cmd.save(self.clean_pdb, 'Prot')
        cmd.save(self.lig_mol2, 'Lig')
        cmd.delete('all')
        print(f'  [{self.name}] PDB {self.pdb}: protein saved, ligand {n_lig_atoms} atoms')
        return n_lig_atoms >= 5

    # ---------- 2. Ligand prep ----------
    def _prep_ligand(self):
        """Add hydrogens to co-crystallized ligand."""
        self.lig_H = self._path(f'{self.pdb}_lig_H.mol2')
        mol = list(pybel.readfile(filename=self.lig_mol2, format='mol2'))[0]
        mol.addh()
        out = pybel.Outputfile(filename=self.lig_H, format='mol2', overwrite=True)
        out.write(mol)
        out.close()
        print(f'  [{self.name}] Ligand H added: {os.path.basename(self.lig_H)}')

    # ---------- 3. Protein prep ----------
    def _prep_protein(self):
        """Fix protein and compute docking box from ligand position."""
        self.prot_H = self._path(f'{self.pdb}_clean_H.pdb')
        fix_protein(self.clean_pdb, self.prot_H, addHs_pH=7.2, try_renumber=True)

        cmd.delete('all')
        cmd.load(self.prot_H, 'prot')
        cmd.load(self.lig_H, 'lig')
        self.center, self.size = getbox('lig', extending=self.extending)
        cmd.delete('all')
        print(f'  [{self.name}] Protein fixed, box center=({self.center["center_x"]:.1f}, '
              f'{self.center["center_y"]:.1f}, {self.center["center_z"]:.1f})')

    # ---------- 4. PDBQT conversion ----------
    def _make_pdbqt(self):
        """Convert receptor and ligand to PDBQT via ADFRsuite."""
        self.rec_qt = self._path(f'{self.pdb}_clean_H.pdbqt')
        self.lig_qt = self._path(f'{self.pdb}_lig_H.pdbqt')
        os.system(f'{BIN_DIR}/prepare_receptor -r {self.prot_H} -o {self.rec_qt} > /dev/null 2>&1')
        os.system(f'{BIN_DIR}/prepare_ligand -l {self.lig_H} -o {self.lig_sdf} > /dev/null 2>&1')
        ok = os.path.exists(self.rec_qt) and os.path.exists(self.lig_qt)
        print(f'  [{self.name}] PDBQT: {"OK" if ok else "FAILED"}')
        return ok

    # ---------- Full prepare ----------
    def prepare(self):
        """Run the full preparation pipeline. Returns True on success."""
        try:
            has_lig = self._fetch_clean()
            if not has_lig:
                print(f'  [{self.name}] WARNING: No ligand found, skipping.')
                return False
            self._prep_ligand()
            self._prep_protein()
            ok = self._make_pdbqt()
            self.prepared = ok
            return ok
        except Exception as e:
            print(f'  [{self.name}] ERROR during preparation: {e}')
            return False

    # ---------- Docking ----------
    def dock_native(self, method='smina', exhaustiveness=64, n_poses=20):
        """
        Dock the co-crystallized ligand back into the binding site.

        Parameters
        ----------
        method : 'smina' or 'vina'
        exhaustiveness : int
        n_poses : int

        Returns
        -------
        sdf_path : str or None
        """
        if not self.prepared:
            print(f'  [{self.name}] Not prepared, call prepare() first.')
            return None
        return self._dock(self.lig_sdf, 'native', method, exhaustiveness, n_poses)

    def dock_sdf(self, sdf_path, label='custom', method='smina',
                 exhaustiveness=32, n_poses=10):
        """
        Dock ligands from an SDF file.
        """
        if not self.prepared:
            return None
        return self._dock(sdf_path, label, method, exhaustiveness, n_poses)

    def dock_smiles(self, smiles, mol_name='query', method='smina',
                    exhaustiveness=32, n_poses=10):
        """
        Dock a single molecule from SMILES string.
        """
        if not self.prepared:
            return None
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f'  [{self.name}] Invalid SMILES: {smiles}')
            return None
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, randomSeed=42)
        AllChem.MMFFOptimizeMolecule(mol)
        sdf_tmp = self._path(f'{mol_name}.sdf')
        w = Chem.SDWriter(sdf_tmp)
        w.write(mol)
        w.close()
        return self._dock(sdf_tmp, mol_name, method, exhaustiveness, n_poses)

    def _dock(self, ligand_file, label, method, exhaustiveness, n_poses):
        """Internal: run docking and store result path."""
        out_sdf = self._path(f'{self.pdb}_{label}_{method}_out.sdf')
        c = self.center
        s = self.size

        if method == 'smina':
            ret = os.system(
                f'{BIN_DIR}/smina -r {self.rec_qt} -l {ligand_file} -o {out_sdf} '
                f'--center_x {c["center_x"]} --center_y {c["center_y"]} --center_z {c["center_z"]} '
                f'--size_x {s["size_x"]} --size_y {s["size_y"]} --size_z {s["size_z"]} '
                f'--exhaustiveness {exhaustiveness} --num_modes {n_poses} > /dev/null 2>&1'
            )
        elif method == 'vina':
            out_pdbqt = out_sdf.replace('.sdf', '.pdbqt')
            v = Vina(sf_name='vina')
            v.set_receptor(self.rec_qt)
            v.set_ligand_from_file(ligand_file)
            v.compute_vina_maps(
                center=[c['center_x'], c['center_y'], c['center_z']],
                box_size=[s['size_x'], s['size_y'], s['size_z']])
            v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
            v.write_poses(out_pdbqt, n_poses=n_poses, overwrite=True)
            pdbqt_to_sdf(out_pdbqt, out_sdf)

        if os.path.exists(out_sdf) and os.path.getsize(out_sdf) > 0:
            self.docking_results[label] = out_sdf
            print(f'  [{self.name}] Docked {label} ({method}): {os.path.basename(out_sdf)}')
            return out_sdf
        else:
            print(f'  [{self.name}] Docking FAILED for {label}')
            return None

    # ---------- Analysis ----------
    def get_scores(self, label='native'):
        """Load docking scores as a DataFrame."""
        sdf = self.docking_results.get(label)
        if not sdf or not os.path.exists(sdf):
            return pd.DataFrame()
        df = PandasTools.LoadSDF(sdf, smilesName='SMILES', molColName='Molecule',
                                 includeFingerprints=False, strictParsing=False)
        if 'Score' in df.columns:
            df = df.rename(columns={'Score': 'minimizedAffinity'})
        if 'minimizedAffinity' in df.columns:
            df['minimizedAffinity'] = pd.to_numeric(df['minimizedAffinity'], errors='coerce')
        df['kinase'] = self.name
        df['pdb'] = self.pdb
        return df

    def get_prolif(self, label='native', pose_indices=None):
        """
        Compute ProLIF interaction fingerprints.

        Returns
        -------
        fp : plf.Fingerprint
        df : DataFrame
        lig_suppl : list of Molecules
        """
        sdf = self.docking_results.get(label)
        if not sdf:
            return None, None, None
        prot_mol = Chem.MolFromPDBFile(self.prot_H, removeHs=False, sanitize=False)
        if prot_mol is None:
            prot_u = mda.Universe(self.prot_H, guess_bonds=True)
            prot_mol = plf.Molecule.from_mda(prot_u)
        else:
            prot_mol = plf.Molecule(prot_mol)
        lig_suppl = list(plf.sdf_supplier(sdf))
        if pose_indices is not None:
            lig_suppl = [lig_suppl[i] for i in pose_indices if i < len(lig_suppl)]
        fp = plf.Fingerprint()
        fp.run_from_iterable(lig_suppl, prot_mol, n_jobs=1)
        df = fp.to_dataframe()
        return fp, df, lig_suppl

    def view_3d(self, label='native', pose_index=0):
        """Interactive 3D viewer with py3Dmol."""
        sdf = self.docking_results.get(label)
        if not sdf:
            print('No docking results.')
            return None
        view = py3Dmol.view(width=800, height=500)
        view.removeAllModels()
        view.setViewStyle({'style': 'outline', 'color': 'black', 'width': 0.1})
        # Protein
        view.addModel(open(self.prot_H).read(), 'pdb')
        view.getModel().setStyle(
            {'cartoon': {'arrows': True, 'tubes': True, 'style': 'oval', 'color': 'white'}})
        view.addSurface(py3Dmol.VDW, {'opacity': 0.5, 'color': 'white'})
        # Reference ligand (magenta)
        if os.path.exists(self.lig_H):
            view.addModel(open(self.lig_H).read(), 'mol2')
            view.getModel().setStyle({}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.15}})
        # Docked pose (green)
        poses = [m for m in Chem.SDMolSupplier(sdf, True) if m is not None]
        if pose_index < len(poses):
            view.addModel(Chem.MolToMolBlock(poses[pose_index]), 'mol')
            view.getModel().setStyle({}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.25}})
        view.zoomTo()
        return view

    def __repr__(self):
        status = 'ready' if self.prepared else 'not prepared'
        n_dock = len(self.docking_results)
        return f'DockingPipeline({self.name}, {self.pdb}, {status}, {n_dock} dockings)'


print('DockingPipeline class defined.')
print('Methods: prepare(), dock_native(), dock_sdf(), dock_smiles(), get_scores(), get_prolif(), view_3d()')

---
## 4. Batch Preparation

Fetch, clean, and prepare all 10 kinases.

In [ ]:
#@title 4-1. Prepare All Kinases {display-mode: "form"}

WORK_ROOT = '/content/docking'
pipelines = {}

for i, k in enumerate(KINASES, 1):
    print(f'\n--- [{i}/10] {k["name"]} ({k["pdb"]}) ---')
    pipe = DockingPipeline(
        name=k['name'], pdb_code=k['pdb'], chain=k['chain'],
        work_dir=os.path.join(WORK_ROOT, k['name'])
    )
    ok = pipe.prepare()
    if ok:
        pipelines[k['name']] = pipe
    else:
        print(f'  SKIPPED: {k["name"]}')

print(f'\n=== {len(pipelines)}/{len(KINASES)} kinases prepared ===')
for name, pipe in pipelines.items():
    print(f'  {pipe}')

---
## 5. Native Re-docking

Dock each kinase's co-crystallized inhibitor back into its own binding site (validation).

In [ ]:
#@title 5-1. Run Re-docking (smina, ~10 min) {display-mode: "form"}

for name, pipe in pipelines.items():
    pipe.dock_native(method='smina', exhaustiveness=64, n_poses=20)

print(f'\n=== Re-docking complete for {len(pipelines)} kinases ===')

### 5-2. Collect Scores

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 5-2. Collect Scores {display-mode: "form"}

all_scores = []
for name, pipe in pipelines.items():
    df = pipe.get_scores('native')
    if not df.empty:
        k_info = next(k for k in KINASES if k['name'] == name)
        df['group'] = k_info['group']
        df['family'] = k_info['family']
        all_scores.append(df)

scores_df = pd.concat(all_scores, ignore_index=True)
print(f'Total docking poses: {len(scores_df)}')

# Best score per kinase
best = scores_df.groupby('kinase')['minimizedAffinity'].min().sort_values()
print('\nBest docking scores (kcal/mol):')
for k, v in best.items():
    print(f'  {k:8s}: {v:.2f}')

---
## 6. Score Analysis & Comparison

In [ ]:
#@title 6-1. Score Distribution by Kinase {display-mode: "form"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot by kinase, sorted by median
order = scores_df.groupby('kinase')['minimizedAffinity'].median().sort_values().index.tolist()
sns.boxplot(data=scores_df, x='kinase', y='minimizedAffinity', order=order,
            palette='Set3', ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_ylabel('Binding Affinity (kcal/mol)')
axes[0].set_title('Native Re-docking Scores by Kinase')
axes[0].axhline(y=-7.0, color='red', linestyle='--', alpha=0.5, label='−7 kcal/mol')
axes[0].legend()

# Violin by kinase group
sns.violinplot(data=scores_df, x='group', y='minimizedAffinity',
               palette='pastel', ax=axes[1], inner='box')
axes[1].set_ylabel('Binding Affinity (kcal/mol)')
axes[1].set_title('Score Distribution by Kinase Group')

plt.tight_layout()
plt.show()

### 6-2. Best Scores Barplot

6-2. Best Scores Barplot


In [ ]:
#@title 6-2. Best Scores Barplot {display-mode: "form"}

best_df = scores_df.groupby(['kinase', 'group']).agg(
    best_score=('minimizedAffinity', 'min'),
    mean_score=('minimizedAffinity', 'mean'),
    n_poses=('minimizedAffinity', 'count')
).reset_index().sort_values('best_score')

group_colors = {'TK': '#E74C3C', 'CMGC': '#3498DB', 'TKL': '#2ECC71',
                'AGC': '#F39C12', 'Other': '#9B59B6'}
colors = [group_colors.get(g, 'gray') for g in best_df['group']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(best_df['kinase'], best_df['best_score'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Best Binding Affinity (kcal/mol)')
ax.set_title('Best Re-docking Score per Kinase')
ax.axvline(x=-7.0, color='red', linestyle='--', alpha=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items()]
ax.legend(handles=legend_elements, title='Kinase Group', loc='lower right')

# Score labels
for bar, score in zip(bars, best_df['best_score']):
    ax.text(bar.get_width() - 0.3, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

best_df[['kinase', 'group', 'best_score', 'mean_score', 'n_poses']]

---
## 7. Cross-docking Selectivity Analysis (Optional)

Dock each kinase's native ligand against ALL kinases → 10 × 10 selectivity matrix.

In [ ]:
#@title 7-1. Cross-dock All Combinations (~30-50 min) {display-mode: "form"}

RUN_CROSSDOCK = True  #@param {type:"boolean"}

cross_scores = {}  # (ligand_from, docked_into) -> best_score

if RUN_CROSSDOCK:
    kinase_names = list(pipelines.keys())
    total = len(kinase_names) ** 2
    done = 0

    for lig_name in kinase_names:
        lig_pipe = pipelines[lig_name]
        # Use the native ligand PDBQT from this kinase
        lig_file = lig_pipe.lig_qt
        if not os.path.exists(lig_file):
            continue

        for target_name in kinase_names:
            target_pipe = pipelines[target_name]
            done += 1
            label = f'xdock_{lig_name}'
            sdf = target_pipe._dock(lig_file, label, 'smina', exhaustiveness=32, n_poses=5)
            if sdf:
                df = target_pipe.get_scores(label)
                if not df.empty and 'minimizedAffinity' in df.columns:
                    cross_scores[(lig_name, target_name)] = df['minimizedAffinity'].min()
            if done % 10 == 0:
                print(f'  Progress: {done}/{total}')

    print(f'\n=== Cross-docking complete: {len(cross_scores)} combinations ===')
else:
    print('Cross-docking skipped. Set RUN_CROSSDOCK = True to enable.')

### 7-2. Selectivity Heatmap

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 7-2. Selectivity Heatmap {display-mode: "form"}

if cross_scores:
    kinase_names = list(pipelines.keys())
    matrix = pd.DataFrame(np.nan, index=kinase_names, columns=kinase_names)

    for (lig, tgt), score in cross_scores.items():
        matrix.loc[tgt, lig] = score

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(matrix.astype(float), annot=True, fmt='.1f', cmap='RdYlGn',
                center=-7.0, ax=ax, linewidths=0.5,
                cbar_kws={'label': 'Binding Affinity (kcal/mol)'})
    ax.set_xlabel('Ligand from')
    ax.set_ylabel('Docked into')
    ax.set_title('Cross-docking Selectivity Matrix\n(rows=target kinase, columns=ligand origin)')
    plt.tight_layout()
    plt.show()

    # Selectivity index: diagonal − off-diagonal mean
    print('\nSelectivity Index (lower = more selective):')
    for name in kinase_names:
        diag = matrix.loc[name, name]
        off_diag = matrix.loc[name, :].drop(name).mean()
        if pd.notna(diag) and pd.notna(off_diag):
            sel = diag - off_diag
            print(f'  {name:8s}: native={diag:.1f}, others_mean={off_diag:.1f}, selectivity={sel:+.1f}')
else:
    print('No cross-docking data. Run section 7-1 first.')

### 7-3. Hierarchical Clustering by Cross-docking Profile

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 7-3. Hierarchical Clustering by Cross-docking Profile {display-mode: "form"}

if cross_scores:
    # Cluster kinases by their cross-docking score profiles
    matrix_filled = matrix.astype(float).fillna(0)
    linkage_matrix = linkage(matrix_filled.values, method='ward')

    fig, ax = plt.subplots(figsize=(10, 5))
    dendrogram(linkage_matrix, labels=matrix_filled.index.tolist(), ax=ax,
               leaf_rotation=45, leaf_font_size=11)
    ax.set_ylabel('Distance (Ward)')
    ax.set_title('Kinase Clustering by Cross-docking Selectivity Profile')
    plt.tight_layout()
    plt.show()
else:
    print('No cross-docking data available.')

---
## 8. Protein-Ligand Interaction Fingerprints (ProLIF)

In [ ]:
#@title 8-1. Generate ProLIF Fingerprints for All Kinases {display-mode: "form"}

prolif_data = {}  # name -> (fp, df, lig_suppl)

for name, pipe in pipelines.items():
    try:
        fp, df, lig_suppl = pipe.get_prolif('native', pose_indices=[0])
        if fp is not None:
            prolif_data[name] = (fp, df, lig_suppl)
            n_interactions = df.any().sum() if df is not None else 0
            print(f'  [{name}] {n_interactions} interactions detected')
    except Exception as e:
        print(f'  [{name}] ProLIF failed: {e}')

print(f'\n=== ProLIF computed for {len(prolif_data)}/{len(pipelines)} kinases ===')

### 8-2. Interaction Type Comparison

ProLIF로 단백질-리간드 상호작용 핑거프린트를 계산합니다.


In [ ]:
#@title 8-2. Interaction Type Comparison {display-mode: "form"}

INTERACTION_TYPES = ['Hydrophobic', 'HBDonor', 'HBAcceptor', 'PiStacking',
                     'PiCation', 'CationPi', 'Cationic', 'Anionic']

interaction_counts = {}
for name, (fp, df, _) in prolif_data.items():
    counts = {}
    if df is not None and not df.empty:
        for itype in INTERACTION_TYPES:
            # Count residues with this interaction type
            cols_with_type = [c for c in df.columns if itype in str(c)]
            counts[itype] = sum(df[cols_with_type].any().sum() for _ in [1]) if cols_with_type else 0
    interaction_counts[name] = counts

int_df = pd.DataFrame(interaction_counts).T.fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 6))
int_df.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='black', linewidth=0.3)
ax.set_ylabel('Number of Interacting Residues')
ax.set_title('Interaction Types per Kinase (Best Docking Pose)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 8-3. Interaction Network for Selected Kinase

ProLIF로 단백질-리간드 상호작용 핑거프린트를 계산합니다.


In [ ]:
#@title 8-3. Interaction Network for Selected Kinase {display-mode: "form"}

KINASE_TO_VIEW = 'EGFR'  #@param {type:"string"}

if KINASE_TO_VIEW in prolif_data:
    fp, df, lig_suppl = prolif_data[KINASE_TO_VIEW]
    net = LigNetwork.from_fingerprint(df, lig_suppl[0],
                               kind='frame', frame=0, rotation=270)
    print(f'Interaction network for {KINASE_TO_VIEW}:')
    net.display()
else:
    print(f'{KINASE_TO_VIEW} not in ProLIF data. Available: {list(prolif_data.keys())}')

---
## 8.5 Advanced Comparative Analysis

Ligand efficiency, IFP similarity, PCA clustering, molecular fingerprint analysis.

In [ ]:
#@title 8.5-1. Ligand Efficiency Comparison {display-mode: "form"}
from rdkit.Chem import Descriptors

le_data = []
for name, pipe in pipelines.items():
    df = pipe.get_scores("native")
    if df.empty or "minimizedAffinity" not in df.columns: continue
    for _, row in df.iterrows():
        mol = row.get("Molecule")
        if mol is not None:
            ha = mol.GetNumHeavyAtoms()
            mw = Descriptors.MolWt(mol)
            score = row["minimizedAffinity"]
            le_data.append({"kinase": name, "score": score,
                            "HA": ha, "MW": round(mw,1),
                            "LE": round(-score/ha, 3) if ha > 0 else 0,
                            "BEI": round(-score/(mw/1000), 2) if mw > 0 else 0,
                            "group": next(k["group"] for k in KINASES if k["name"]==name)})

le_df = pd.DataFrame(le_data)
if not le_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # LE by kinase
    sns.boxplot(data=le_df, x="kinase", y="LE", palette="Set3", ax=axes[0])
    axes[0].axhline(y=0.3, color="green", linestyle="--", label="LE=0.3")
    axes[0].set_title("Ligand Efficiency by Kinase")
    axes[0].legend()
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha="right")
    # LE vs Score scatter by group
    for grp in le_df["group"].unique():
        sub = le_df[le_df["group"]==grp]
        axes[1].scatter(sub["score"], sub["LE"], label=grp, s=40, alpha=0.7)
    axes[1].set_xlabel("Score (kcal/mol)"); axes[1].set_ylabel("LE")
    axes[1].set_title("Score vs Ligand Efficiency"); axes[1].legend()
    plt.tight_layout(); plt.show()


### 8.5-2. IFP Similarity Matrix & PCA

8.5-2. IFP Similarity Matrix & PCA


In [ ]:
#@title 8.5-2. IFP Similarity Matrix & PCA {display-mode: "form"}
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Build binary IFP vectors for each kinase (best pose)
ifp_vectors = {}
all_interactions = set()
for name, (fp, df, _) in prolif_data.items():
    if df is not None and not df.empty:
        bv = df.iloc[0].to_dict()
        ifp_vectors[name] = bv
        all_interactions.update(bv.keys())

if len(ifp_vectors) >= 3:
    # Align to common interaction set
    all_int = sorted(all_interactions)
    mat = pd.DataFrame({name: {k: int(bool(v.get(k, False))) for k in all_int}
                        for name, v in ifp_vectors.items()}).T
    
    # Tanimoto similarity (RDKit DataStructs)
    from rdkit import DataStructs
    from rdkit.DataStructs import ExplicitBitVect
    n_cols = mat.shape[1]
    fps = []
    for _, row in mat.iterrows():
        bv = ExplicitBitVect(n_cols)
        for idx in range(n_cols):
            if row.iloc[idx]: bv.SetBit(idx)
        fps.append(bv)
    n = len(fps)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i][j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])
    sim_df = pd.DataFrame(sim, index=mat.index, columns=mat.index)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[0],
               vmin=0, vmax=1, linewidths=0.5)
    axes[0].set_title("IFP Tanimoto Similarity")
    
    # PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(mat.values)
    groups = [next((k["group"] for k in KINASES if k["name"]==n), "?") for n in mat.index]
    group_colors_map = {"TK": "#E74C3C", "CMGC": "#3498DB", "TKL": "#2ECC71",
                        "AGC": "#F39C12", "Other": "#9B59B6"}
    for n, (x, y), g in zip(mat.index, coords, groups):
        axes[1].scatter(x, y, color=group_colors_map.get(g, "gray"), s=100, zorder=5)
        axes[1].annotate(n, (x, y), fontsize=9, ha="center", va="bottom")
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%})")
    axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
    axes[1].set_title("PCA of Interaction Fingerprints")
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color=c, label=g) for g,c in group_colors_map.items()],
                   loc="best")
    plt.tight_layout(); plt.show()
else:
    print(f"Need >=3 kinases with ProLIF data, have {len(ifp_vectors)}")


### 8.5-3. Ligand Molecular Fingerprint Clustering (ECFP4)

ProLIF로 단백질-리간드 상호작용 핑거프린트를 계산합니다.


In [ ]:
#@title 8.5-3. Ligand Molecular Fingerprint Clustering (ECFP4) {display-mode: "form"}

# Compute ECFP4 for each co-crystallized ligand
fp_dict = {}
for name, pipe in pipelines.items():
    sdf = pipe.docking_results.get("native")
    if sdf:
        suppl = Chem.SDMolSupplier(sdf, True)
        if len(suppl) > 0 and suppl[0] is not None:
            fp_dict[name] = AllChem.GetMorganFingerprintAsBitVect(suppl[0], 2, nBits=2048)

if len(fp_dict) >= 3:
    names = list(fp_dict.keys())
    fps = [fp_dict[n] for n in names]
    
    # Tanimoto similarity matrix
    n = len(fps)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])
    sim_df = pd.DataFrame(sim, index=names, columns=names)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[0],
               vmin=0, vmax=1, linewidths=0.5)
    axes[0].set_title("ECFP4 Tanimoto Similarity (Ligands)")
    
    # Dendrogram
    from scipy.cluster.hierarchy import linkage, dendrogram
    from scipy.spatial.distance import squareform
    dist = 1 - sim
    np.fill_diagonal(dist, 0)
    Z = linkage(squareform(dist), method="average")
    dendrogram(Z, labels=names, ax=axes[1], leaf_rotation=45, leaf_font_size=10)
    axes[1].set_ylabel("Tanimoto Distance")
    axes[1].set_title("Ligand Structural Clustering")
    plt.tight_layout(); plt.show()


### 8.5-4. Interaction Frequency: Conserved Contacts

ProLIF로 단백질-리간드 상호작용 핑거프린트를 계산합니다.


In [ ]:
#@title 8.5-4. Interaction Frequency: Conserved Contacts {display-mode: "form"}

# For each kinase, get interacting residues from best pose
interaction_map = defaultdict(lambda: defaultdict(int))
for name, (fp, df, _) in prolif_data.items():
    if df is None or df.empty: continue
    for col in df.columns:
        if df.iloc[0][col]:
            try:
                res = str(col[1])  # protein residue
                itype = str(col[2])  # interaction type
                interaction_map[itype][name] += 1
            except (IndexError, TypeError):
                pass

# Count how many kinases each interaction type appears in
print("Interaction type prevalence across kinases:")
print("-" * 45)
for itype in sorted(interaction_map.keys()):
    n_kinases = len(interaction_map[itype])
    avg_count = np.mean(list(interaction_map[itype].values()))
    print(f"  {itype:15s}: {n_kinases}/{len(prolif_data)} kinases, avg {avg_count:.1f} contacts")


---
## 9. 3D Visualization

In [ ]:
#@title 9-1. Interactive 3D Viewer {display-mode: "form"}

VIEW_KINASE = 'EGFR'  #@param {type:"string"}
POSE_INDEX = 0  #@param {type:"integer"}

if VIEW_KINASE in pipelines:
    print(f'{VIEW_KINASE}: magenta=reference ligand, green=docked pose #{POSE_INDEX}')
    pipelines[VIEW_KINASE].view_3d('native', POSE_INDEX)
else:
    print(f'{VIEW_KINASE} not found. Available: {list(pipelines.keys())}')

### 9-2. Gallery: Best Pose per Kinase (2D)

도킹 결과를 3D로 시각화합니다.


In [ ]:
#@title 9-2. Gallery: Best Pose per Kinase (2D) {display-mode: "form"}

mols_2d = []
legends = []
for name, pipe in pipelines.items():
    sdf = pipe.docking_results.get('native')
    if sdf:
        suppl = Chem.SDMolSupplier(sdf, True)
        mol = suppl[0] if len(suppl) > 0 else None
        if mol:
            mol_2d = Chem.RemoveHs(mol)
            mol_2d.RemoveAllConformers()
            AllChem.Compute2DCoords(mol_2d)
            mols_2d.append(mol_2d)
            df = pipe.get_scores('native')
            best = df['minimizedAffinity'].min() if 'minimizedAffinity' in df.columns else 0
            legends.append(f'{name}\n{best:.1f} kcal/mol')

if mols_2d:
    img = Draw.MolsToGridImage(mols_2d, legends=legends,
                                molsPerRow=5, subImgSize=(300, 200))
    img

---
## 10. Binding Site Structural Comparison

In [ ]:
#@title 10-1. Binding Site Properties {display-mode: "form"}

site_props = []
for name, pipe in pipelines.items():
    if not pipe.prepared:
        continue
    cmd.delete('all')
    cmd.load(pipe.prot_H, 'prot')
    cmd.load(pipe.lig_H, 'lig')
    # Select residues within 5A of ligand
    cmd.select('site', 'prot within 5.0 of lig')
    n_atoms = cmd.count_atoms('site')
    n_res = cmd.count_atoms('site and name CA')

    # Count charged, aromatic, hydrophobic residues
    n_charged = cmd.count_atoms('site and name CA and (resn ASP+GLU+LYS+ARG+HIS)')
    n_aromatic = cmd.count_atoms('site and name CA and (resn PHE+TYR+TRP+HIS)')
    n_hydrophobic = cmd.count_atoms('site and name CA and (resn ALA+VAL+LEU+ILE+MET+PRO)')
    n_polar = cmd.count_atoms('site and name CA and (resn SER+THR+ASN+GLN+CYS)')

    site_props.append({
        'kinase': name,
        'group': next(k['group'] for k in KINASES if k['name'] == name),
        'site_residues': n_res,
        'charged': n_charged,
        'aromatic': n_aromatic,
        'hydrophobic': n_hydrophobic,
        'polar': n_polar,
    })
    cmd.delete('all')

site_df = pd.DataFrame(site_props).set_index('kinase')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: binding site composition
comp_cols = ['charged', 'aromatic', 'hydrophobic', 'polar']
site_df[comp_cols].plot(kind='bar', stacked=True, ax=axes[0],
                         color=['#E74C3C', '#9B59B6', '#3498DB', '#2ECC71'],
                         edgecolor='black', linewidth=0.3)
axes[0].set_ylabel('Number of Residues')
axes[0].set_title('Binding Site Composition (within 5\u00c5)')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Binding site size
site_sorted = site_df.sort_values('site_residues')
colors = [group_colors.get(g, 'gray') for g in site_sorted['group']]
axes[1].barh(site_sorted.index, site_sorted['site_residues'],
             color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Number of Binding Site Residues')
axes[1].set_title('Binding Site Size Comparison')

plt.tight_layout()
plt.show()

site_df

### 10-2. Kinase Clustering by Binding Site Composition

구조적 유사도를 기반으로 클러스터링합니다.


In [ ]:
#@title 10-2. Kinase Clustering by Binding Site Composition {display-mode: "form"}

# Normalize binding site features
from sklearn.preprocessing import StandardScaler

features = site_df[comp_cols].values
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

linkage_site = linkage(features_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dendrogram
dendrogram(linkage_site, labels=site_df.index.tolist(), ax=axes[0],
           leaf_rotation=45, leaf_font_size=10)
axes[0].set_ylabel('Distance (Ward)')
axes[0].set_title('Kinase Clustering by Binding Site Properties')

# Clustermap-style heatmap
site_norm = pd.DataFrame(features_scaled, index=site_df.index, columns=comp_cols)
sns.heatmap(site_norm, annot=True, fmt='.1f', cmap='coolwarm', center=0,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Normalized Binding Site Features')

plt.tight_layout()
plt.show()

---
## 11. Custom Ligand Docking

Dock your own molecules against any kinase in the panel.

In [ ]:
#@title 11-1. Dock SMILES Against a Kinase {display-mode: "form"}

TARGET_KINASE = 'EGFR'  #@param {type:"string"}
SMILES = 'c1ccc2c(c1)c1ccccc1[nH]2'  #@param {type:"string"}
MOL_NAME = 'my_compound'  #@param {type:"string"}

if TARGET_KINASE in pipelines:
    sdf = pipelines[TARGET_KINASE].dock_smiles(SMILES, MOL_NAME)
    if sdf:
        df = pipelines[TARGET_KINASE].get_scores(MOL_NAME)
        print(f'\nBest score: {df["minimizedAffinity"].min():.2f} kcal/mol')
        pipelines[TARGET_KINASE].view_3d(MOL_NAME, 0)
else:
    print(f'{TARGET_KINASE} not available.')

### 11-2. Selectivity Screen: One Ligand vs All Kinases

Ligand Efficiency (LE)와 BEI를 계산하여 분자 크기 대비 결합 효율을 평가합니다.


In [ ]:
#@title 11-2. Selectivity Screen: One Ligand vs All Kinases {display-mode: "form"}

SCREEN_SMILES = 'c1ccc2c(c1)c1ccccc1[nH]2'  #@param {type:"string"}
SCREEN_NAME = 'screen_query'  #@param {type:"string"}

screen_results = []
for name, pipe in pipelines.items():
    sdf = pipe.dock_smiles(SCREEN_SMILES, SCREEN_NAME, exhaustiveness=32, n_poses=5)
    if sdf:
        df = pipe.get_scores(SCREEN_NAME)
        if not df.empty:
            screen_results.append({
                'kinase': name,
                'best_score': df['minimizedAffinity'].min(),
                'group': next(k['group'] for k in KINASES if k['name'] == name)
            })

if screen_results:
    screen_df = pd.DataFrame(screen_results).sort_values('best_score')

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [group_colors.get(g, 'gray') for g in screen_df['group']]
    ax.barh(screen_df['kinase'], screen_df['best_score'], color=colors, edgecolor='black')
    ax.set_xlabel('Binding Affinity (kcal/mol)')
    ax.set_title(f'Selectivity Profile: {SCREEN_NAME}')
    plt.tight_layout()
    plt.show()

    print(screen_df.to_string(index=False))

---
## 12. Summary & Export

In [ ]:
#@title 12-1. Summary Table {display-mode: "form"}

summary = []
for k in KINASES:
    name = k['name']
    row = {
        'Kinase': name,
        'PDB': k['pdb'],
        'Group': k['group'],
        'Family': k['family'],
        'Prepared': name in pipelines,
    }
    if name in pipelines:
        df = pipelines[name].get_scores('native')
        if not df.empty and 'minimizedAffinity' in df.columns:
            row['Best Score'] = f"{df['minimizedAffinity'].min():.2f}"
            row['Mean Score'] = f"{df['minimizedAffinity'].mean():.2f}"
            row['N Poses'] = len(df)
        if name in prolif_data:
            _, idf, _ = prolif_data[name]
            row['N Interactions'] = idf.any().sum() if idf is not None else 0
    summary.append(row)

summary_df = pd.DataFrame(summary)
summary_df

### 12-2. Export All Results

결과 파일을 다운로드합니다.


In [ ]:
#@title 12-2. Export All Results {display-mode: "form"}
import shutil

# Save scores CSV
scores_csv = f'{WORK_ROOT}/all_docking_scores.csv'
scores_df.drop(columns=['Molecule'], errors='ignore').to_csv(scores_csv, index=False)
print(f'Scores CSV: {scores_csv}')

# Save summary
summary_csv = f'{WORK_ROOT}/summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Summary CSV: {summary_csv}')

# Save cross-docking matrix
if cross_scores:
    matrix_csv = f'{WORK_ROOT}/crossdock_matrix.csv'
    matrix.to_csv(matrix_csv)
    print(f'Cross-docking matrix: {matrix_csv}')

# Save interaction networks as HTML
for name, (fp, df, lig_suppl) in prolif_data.items():
    html_path = f'{WORK_ROOT}/{name}/{name}_interactions.html'
    try:
        net = LigNetwork.from_fingerprint(df, lig_suppl[0], kind='frame', frame=0, rotation=270)
        net.save(html_path)
    except Exception:
        pass
print(f'Interaction HTMLs saved per kinase.')

# Zip everything
zip_path = '/content/kinase_panel_results'
shutil.make_archive(zip_path, 'zip', WORK_ROOT)
print(f'\nArchive: {zip_path}.zip')

try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print('Not in Colab — results at:', WORK_ROOT)